<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/13_app_buscador.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13 · App web del buscador (Streamlit)

Convierte el buscador de los notebooks 09-12 en una **página web**: se escribe una descripción en español o inglés
y aparecen los pósters y títulos de las películas más parecidas.

| Sección | Qué hace | Resultado |
|---|---|---|
| 1 | Exporta lo que necesita la app: catálogo (MM-IMDB + películas 2020-2026), miniaturas, vectores (sin entrenar y entrenados) y cifras de la comparación temporal | carpeta `app_buscador/` en Drive |
| 2 | Escribe el código de la app (`app.py`) y sus archivos de configuración | `app.py`, `requirements.txt`, `Dockerfile` |
| 3 | Revisa que la búsqueda dé lo mismo que en el notebook 11 | — |
| 4 | **Lanza la app desde Colab** con un link público temporal | link `https://….trycloudflare.com` |
| 5 | (Opcional) La publica de forma **permanente** en Hugging Face Spaces | link `https://huggingface.co/spaces/…` |

**Requisitos:** notebooks 05, 09, 10 y 11 (usa `posters_mini.parquet`, `e5_txt.npy`, `poster_adapt.npy` y `e5_adapt.npy`).
**Opcional:** con los notebooks 07 y 12 la app suma las **películas de 2020 a 2026** de TMDB; con el 08, muestra la
comparación temporal en la pestaña de resultados. Si faltan, la app funciona igual sin ellas.
Corre en CPU, no hace falta GPU.

In [2]:
import os, sys, json, time, re, subprocess
import numpy as np, pandas as pd

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    !pip install -q streamlit sentence-transformers

BASE = os.environ.get('PM_BASE', '/content/drive/MyDrive/proyecto_multimodal')
DATA, EMB = f'{BASE}/data', f'{BASE}/embeddings'
if not os.path.exists(f'{EMB}/clip_img.npy') and os.path.exists(f'{EMB}/embeddings/clip_img.npy'):
    EMB = f'{EMB}/embeddings'
APP_DIR = f'{BASE}/app_buscador'            # todo lo que necesita la app queda aquí
os.makedirs(APP_DIR, exist_ok=True)
print('Vectores en:', EMB, '| App en:', APP_DIR)

Mounted at /content/drive
Vectores en: /content/drive/MyDrive/proyecto_multimodal/embeddings | App en: /content/drive/MyDrive/proyecto_multimodal/app_buscador


## 1. Exportar los datos de la app

Solo entran las **películas de cine** (las mismas del catálogo de búsqueda de los notebooks 10 y 11) y, si están
los archivos del notebook 12, las **películas de 2020 a 2026** de TMDB. Las del grupo de control no se agregan otra vez:
ya están en MM-IMDB.
Los vectores se guardan en `float16` para que pesen la mitad; la pérdida de precisión no cambia el orden de los resultados.

In [3]:
def l2(X):
    X = X.astype(np.float32)
    return X / np.linalg.norm(X, axis=1, keepdims=True).clip(min=1e-8)

pelis = pd.read_parquet(f'{DATA}/mmimdb_con_anio.parquet')
mini = pd.read_parquet(f'{DATA}/posters_mini.parquet')
assert (pelis['fila_hdf5'].values == np.arange(len(pelis))).all() and len(mini) == len(pelis)

filas = np.where(pelis['es_pelicula'].fillna(False).values.astype(bool))[0]
cat = pd.DataFrame({
    'fila_hdf5': filas,
    'tconst': pelis['tconst'].values[filas],
    'titulo': pelis['titulo'].fillna('(sin título)').values[filas],
    'anio': pelis['anio'].values[filas],
    'generos': pelis['generos_imdb_hoy'].fillna('').values[filas],
    'sinopsis': pelis['sinopsis'].fillna('').values[filas],
    'jpg': mini['jpg'].values[filas],
    'fuente': 'MM-IMDB',
})

vectores = {
    'poster_orig.npy': l2(np.load(f'{EMB}/clip_img.npy')),
    'sinopsis_orig.npy': l2(np.load(f'{EMB}/e5_txt.npy')),
    'poster_adapt.npy': l2(np.load(f'{EMB}/poster_adapt.npy')),
    'sinopsis_adapt.npy': l2(np.load(f'{EMB}/e5_adapt.npy')),
}
for nombre, Vx in vectores.items():
    assert len(Vx) == len(pelis), nombre
    vectores[nombre] = Vx[filas]

# Películas de 2020 a 2026 (notebooks 07 y 12). Mismos codificadores y mismos adaptadores del 11.
RECIENTES = {'meta': f'{DATA}/tmdb_meta.parquet', 'mini': f'{DATA}/tmdb_posters_mini.parquet',
             'poster_orig.npy': f'{EMB}/tmdb_clip_img.npy', 'sinopsis_orig.npy': f'{EMB}/tmdb_e5_txt.npy',
             'poster_adapt.npy': f'{EMB}/tmdb_poster_adapt.npy', 'sinopsis_adapt.npy': f'{EMB}/tmdb_e5_adapt.npy'}
CON_RECIENTES = all(os.path.exists(r) for r in RECIENTES.values())
if CON_RECIENTES:
    tm = pd.read_parquet(RECIENTES['meta'])
    mini_tm = pd.read_parquet(RECIENTES['mini'])
    assert (mini_tm['tconst'].values == tm['tconst'].values).all()
    rec = np.where((tm['grupo'] == 'reciente') & ~tm['tconst'].isin(cat['tconst']))[0]   # el control ya está
    cols_hoy = [c for c in tm.columns if c.startswith('hoy_')]
    generos_tm = [','.join(c[4:] for c in cols_hoy if fila[c] == 1) for _, fila in tm[cols_hoy].iterrows()]
    cat = pd.concat([cat, pd.DataFrame({
        'fila_hdf5': -1,
        'tconst': tm['tconst'].values[rec],
        'titulo': tm['titulo'].fillna('(sin título)').values[rec],
        'anio': tm['anio'].astype(float).values[rec],
        'generos': np.array(generos_tm)[rec],
        'sinopsis': tm['overview'].fillna('').values[rec],     # texto original de TMDB, para mostrar
        'jpg': mini_tm['jpg'].values[rec],
        'fuente': 'TMDB',
    })], ignore_index=True)
    for nombre in vectores:
        Vt = l2(np.load(RECIENTES[nombre]))
        assert len(Vt) == len(tm), nombre
        vectores[nombre] = np.concatenate([vectores[nombre], Vt[rec]])
    print(f'Películas 2020-2026 agregadas: {len(rec):,}')
else:
    print('Sin películas 2020-2026 (faltan archivos de los notebooks 07 y 12):',
          [r for r in RECIENTES.values() if not os.path.exists(r)])

cat.to_parquet(f'{APP_DIR}/catalogo.parquet', index=False)
for nombre, Vx in vectores.items():
    assert len(Vx) == len(cat), nombre
    np.save(f'{APP_DIR}/{nombre}', Vx.astype(np.float16))

# Cifras de la comparación temporal (notebooks 08 y 12) para la pestaña de resultados
temporal = {}
RES_CLASIF, RES_BUSQ = f'{BASE}/results', f'{BASE}/resultados'
if os.path.exists(f'{RES_CLASIF}/exp3_descomposicion.csv'):
    e = pd.read_csv(f'{RES_CLASIF}/exp3_descomposicion.csv').query("efecto == 'epoca'").set_index('medida')
    temporal['clasificacion'] = {m: {'diferencia': float(e.loc[m, 'diferencia']),
                                     'ic': [float(e.loc[m, 'ic_inf']), float(e.loc[m, 'ic_sup'])]}
                                 for m in ['ap_imagen', 'ap_texto', 'ap_fusion']}
if os.path.exists(f'{RES_CLASIF}/exp3_por_semilla.csv'):
    ps = pd.read_csv(f'{RES_CLASIF}/exp3_por_semilla.csv')
    temporal['dependencia'] = ps.groupby('grupo')['indice_dependencia_imagen'].mean().round(3).to_dict()
if os.path.exists(f'{RES_BUSQ}/recientes_resumen.json'):
    with open(f'{RES_BUSQ}/recientes_resumen.json') as fh:
        temporal['busqueda'] = json.load(fh)
print('Cifras de la comparación temporal:', list(temporal) or 'ninguna (faltan los notebooks 08 y 12)')

anios = cat['anio'].dropna().astype(int)
config = {'modelo_poster': 'sentence-transformers/clip-ViT-B-32-multilingual-v1',
          'modelo_sinopsis': 'intfloat/multilingual-e5-base',
          'alfa': 0.4,                      # mejor peso del póster en el notebook 11
          'anio_min': int(anios.min()), 'anio_max': int(anios.max()),
          'n_peliculas': int(len(cat)), 'n_recientes': int((cat['fuente'] == 'TMDB').sum()),
          'temporal': temporal,
          # cómo estandarizar los puntajes cuando el catálogo mezcla MM-IMDB y TMDB (elegido en el notebook 12)
          'correccion': temporal.get('busqueda', {}).get('correccion_app', 'global') if CON_RECIENTES else 'global'}
print('Corrección de escala entre fuentes:', config['correccion'])
with open(f'{APP_DIR}/config.json', 'w') as fh:
    json.dump(config, fh, indent=2)

# Figura UMAP (antes vs. después) para la pestaña de resultados: se toma del notebook 11 publicado en GitHub
import base64, urllib.request
try:
    url_nb11 = ('https://raw.githubusercontent.com/AndresAvella09/Proyecto_Multimodal/main/'
                'notebooks/11_buscador_adaptado.ipynb')
    nb11 = json.loads(urllib.request.urlopen(url_nb11, timeout=30).read())
    celda = next(c for c in nb11['cells'] if c['cell_type'] == 'code' and 'umap.UMAP' in ''.join(c['source']))
    png = next(o['data']['image/png'] for o in celda['outputs'] if 'image/png' in o.get('data', {}))
    with open(f'{APP_DIR}/umap.png', 'wb') as fh:
        fh.write(base64.b64decode(png))
    print('Figura UMAP copiada del notebook 11')
except Exception as e:
    print('No se pudo copiar la figura UMAP (la pestaña de resultados la omitirá):', e)

print(f'{len(cat):,} películas | años {config["anio_min"]}-{config["anio_max"]}')
for f in sorted(os.listdir(APP_DIR)):
    print(f'  {f:<22} {os.path.getsize(f"{APP_DIR}/{f}") / 1e6:7.1f} MB')

Películas 2020-2026 agregadas: 2,997
Cifras de la comparación temporal: ['clasificacion', 'dependencia', 'busqueda']
Corrección de escala entre fuentes: por_fuente
Figura UMAP copiada del notebook 11
26,544 películas | años 1912-2026
  .streamlit                 0.0 MB
  Dockerfile                 0.0 MB
  app.py                     0.0 MB
  catalogo.parquet         223.3 MB
  config.json                0.0 MB
  poster_adapt.npy          27.2 MB
  poster_orig.npy           27.2 MB
  requirements.txt           0.0 MB
  sinopsis_adapt.npy        40.8 MB
  sinopsis_orig.npy         40.8 MB
  umap.png                   0.4 MB


## 2. El código de la app

`app.py` es una app de **Streamlit** con estética de cine (negro, blanco y rojo, cintas de película y cámaras antiguas):

- **Portada** con un botón de bienvenida y los créditos del equipo en una esquina.
- **Buscador:** caja de búsqueda, ejemplos y un panel de **opciones** (buscar por ambos / póster / sinopsis; peso del póster,
  que arranca en 0.4 porque fue el que mostró los mejores resultados en el notebook 11; representación entrenada o sin entrenar;
  años y número de resultados). Debajo de cada póster hay un botón **Ver sinopsis**.
- **Arquitectura:** diagrama de los dos caminos (póster y sinopsis) y ficha de cada modelo.
- **Resultados:** las cifras y gráficas de los notebooks 04, 10 y 11, y el mapa UMAP antes vs. después.
- **Explicación técnica:** los conceptos del curso (representaciones, Transformers, aprendizaje contrastivo, adaptador,
  métricas) explicados de forma sencilla.

En modo "ambos", cada resultado dice si lo encontró **más por el póster o más por la sinopsis**: es el desbalance entre
modalidades que estudia el proyecto, visto película por película.

La celda escribe también `requirements.txt`, el tema de colores y un `Dockerfile` para publicarla en Hugging Face (sección 5).

In [4]:
APP = r'''"""Buscador semántico de películas (Streamlit) · Proyecto de Deep Representation Learning.

Póster: CLIP ViT-B/32 multilingüe · Sinopsis: multilingual-e5-base · Adaptadores entrenados (notebook 11).
Los archivos de datos (catalogo.parquet, *.npy, config.json, umap.png) se buscan en BUSCADOR_DIR
o, si no está definida, en la misma carpeta que este archivo.
"""
import html
import io
import json
import os
import re

import altair as alt
import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image

DIR = os.environ.get('BUSCADOR_DIR', os.path.dirname(os.path.abspath(__file__)))
ROJO, BLANCO, GRIS = '#e50914', '#f5f5f5', '#8a8a8a'

st.set_page_config(page_title='Buscador de películas', page_icon='🎬', layout='wide',
                   initial_sidebar_state='collapsed')

# ----------------------------------------------------------------------------- estilo
CINTA_SVG = ("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='34' height='40'>"
             "<rect width='34' height='40' fill='%23000'/>"
             "<rect x='9' y='4' width='16' height='9' rx='2.5' fill='%23f5f5f5'/>"
             "<rect x='9' y='27' width='16' height='9' rx='2.5' fill='%23f5f5f5'/></svg>")

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Bebas+Neue&family=Inter:wght@400;500;600;700&display=swap');
html, body, [class*="css"], .stMarkdown, .stText, p, li, label, input, button {font-family: 'Inter', sans-serif;}
.stApp {background: #000;}
.block-container {padding-top: 1.2rem; max-width: 1200px;}
h1, h2, h3, .marquesina {font-family: 'Bebas Neue', sans-serif !important; letter-spacing: 0.04em; font-weight: 400 !important;}
h2 {font-size: 2.2rem !important;} h3 {font-size: 1.6rem !important;}
[data-testid="stSidebar"], [data-testid="collapsedControl"], [data-testid="stSidebarCollapsedControl"] {display: none;}

/* cinta de película */
.cinta {height: 40px; background-image: url("CINTA_SVG"); background-repeat: repeat-x;
        border-top: 1px solid #222; border-bottom: 1px solid #222; margin: 0.4rem 0 1.2rem 0;}
.cinta.delgada {height: 40px; opacity: 0.85; margin: 0.2rem 0 0.8rem 0;}

/* portada */
.portada {text-align: center; padding: 1.5rem 0 0.5rem 0;}
.portada .marquesina {font-size: 5.2rem; line-height: 0.95; color: #f5f5f5; margin: 0.6rem 0 0.2rem 0;}
.portada .marquesina span {color: #e50914;}
.portada .lema {font-size: 1.15rem; color: #cfcfcf; margin-bottom: 0.4rem;}
.portada .detalle {font-size: 0.9rem; color: #8a8a8a;}
.camaras {display: flex; justify-content: center; align-items: flex-end; gap: 3.5rem; margin-top: 0.5rem;}
.creditos {position: fixed; right: 14px; bottom: 10px; font-size: 10.5px; color: #8a8a8a; text-align: right;
           line-height: 1.35; z-index: 1000; letter-spacing: 0.02em;}
.creditos b {color: #cfcfcf; font-weight: 600;}

/* botones */
.stButton > button {border-radius: 2px; border: 1px solid #e50914; background: #0a0a0a; color: #f5f5f5;
                    font-weight: 500; transition: all 0.15s;}
.stButton > button:hover {background: #e50914; color: #fff; border-color: #e50914;}
.st-key-entrar button {font-family: 'Bebas Neue', sans-serif !important; letter-spacing: 0.08em;
                    padding: 0.55rem 2.5rem; background: #e50914 !important; color: #fff !important; min-height: 3.2rem;}
.st-key-entrar button p {font-family: 'Bebas Neue', sans-serif !important; font-size: 1.8rem !important;
                         white-space: nowrap; overflow: visible; text-overflow: clip;}
.st-key-entrar button:hover {background: #b20710 !important;}

/* pestañas */
.stTabs [data-baseweb="tab-list"] {gap: 0.6rem; border-bottom: 1px solid #222;}
.stTabs [data-baseweb="tab"] {font-family: 'Bebas Neue', sans-serif; font-size: 1.35rem; letter-spacing: 0.06em;
                              color: #bdbdbd; padding: 0.3rem 0.9rem;}
.stTabs [aria-selected="true"] {color: #fff !important;}

/* panel de opciones */
.opcion-nota {font-size: 0.78rem; color: #9a9a9a; line-height: 1.3; margin-top: -0.4rem;}
.opcion-nota b {color: #e50914; font-weight: 600;}

/* tarjetas de resultados */
.tarjeta-titulo {font-weight: 600; font-size: 0.9rem; line-height: 1.2; margin-top: 0.35rem; color: #f5f5f5;}
.tarjeta-meta {font-size: 0.76rem; color: #9a9a9a; line-height: 1.25;}
.etiqueta {display: inline-block; font-size: 0.68rem; padding: 1px 8px; border-radius: 2px; margin: 4px 0 2px 0;
           border: 1px solid #e50914; color: #e50914; text-transform: uppercase; letter-spacing: 0.04em;}
.etiqueta.sinopsis {border-color: #f5f5f5; color: #f5f5f5;}
.etiqueta.estreno {border-color: #f59e0b; color: #f59e0b;}
[data-testid="stImage"] img {border: 3px solid #111; outline: 1px solid #2a2a2a;}

/* sin pantalla completa en las imágenes */
[data-testid="stElementToolbar"], [data-testid="StyledFullScreenButton"],
button[title="View fullscreen"], button[aria-label="View fullscreen"],
[data-testid="stImage"] button, [data-testid="stImageContainer"] button {display: none !important;}

/* bloques de explicación */
.ficha {border: 1px solid #262626; border-left: 3px solid #e50914; padding: 0.8rem 1rem; margin-bottom: 0.8rem;
        background: #070707;}
.ficha h4 {font-family: 'Bebas Neue', sans-serif; font-size: 1.35rem; letter-spacing: 0.05em; margin: 0 0 0.3rem 0;
           color: #fff; font-weight: 400;}
.ficha p, .ficha li {font-size: 0.92rem; color: #d6d6d6; margin-bottom: 0.3rem;}
.kpi {border: 1px solid #262626; padding: 0.8rem 1rem; text-align: center; background: #070707;}
.kpi .valor {font-family: 'Bebas Neue', sans-serif; font-size: 2.3rem; color: #e50914; line-height: 1.1;
             white-space: nowrap;}
.kpi .texto {font-size: 0.82rem; color: #bdbdbd;}

/* diagrama de arquitectura */
.flujo {display: flex; align-items: stretch; gap: 0.5rem; margin: 0.4rem 0 0.9rem 0; flex-wrap: wrap;}
.caja {flex: 1; min-width: 150px; border: 1px solid #333; background: #0b0b0b; padding: 0.6rem 0.7rem;}
.caja.roja {border-color: #e50914;}
.caja .nombre {font-family: 'Bebas Neue', sans-serif; font-size: 1.2rem; letter-spacing: 0.05em; color: #fff;}
.caja .det {font-size: 0.78rem; color: #a8a8a8; line-height: 1.3;}
.flecha {display: flex; align-items: center; color: #e50914; font-size: 1.4rem;}
.carril {font-size: 0.75rem; color: #e50914; text-transform: uppercase; letter-spacing: 0.1em; margin-top: 0.6rem;}
</style>
""".replace('CINTA_SVG', CINTA_SVG)
st.markdown(CSS, unsafe_allow_html=True)


def cinta(delgada=False):
    st.markdown(f'<div class="cinta{" delgada" if delgada else ""}"></div>', unsafe_allow_html=True)


def camara_svg(ancho=150, espejo=False):
    """Cámara de cine antigua con dos carretes (dibujo propio, genérico)."""
    giro = ' transform="scale(-1,1) translate(-200,0)"' if espejo else ''
    radios = ''.join(
        f'<line x1="{cx}" y1="44" x2="{cx + 20 * np.cos(a):.1f}" y2="{44 + 20 * np.sin(a):.1f}" '
        f'stroke="#f5f5f5" stroke-width="3"/>'
        for cx in (62, 128) for a in np.linspace(0, 2 * np.pi, 6)[:-1])
    return (f'<svg width="{ancho}" viewBox="0 0 200 175" xmlns="http://www.w3.org/2000/svg"><g{giro}>'
            '<circle cx="62" cy="44" r="32" fill="#000" stroke="#f5f5f5" stroke-width="4"/>'
            '<circle cx="128" cy="44" r="32" fill="#000" stroke="#f5f5f5" stroke-width="4"/>'
            f'{radios}'
            '<circle cx="62" cy="44" r="7" fill="#e50914"/><circle cx="128" cy="44" r="7" fill="#e50914"/>'
            '<rect x="38" y="80" width="112" height="56" rx="5" fill="#e50914" stroke="#f5f5f5" stroke-width="4"/>'
            '<rect x="52" y="92" width="34" height="18" rx="2" fill="#000"/>'
            '<polygon points="150,96 188,82 188,134 150,120" fill="#000" stroke="#f5f5f5" stroke-width="4"/>'
            '<line x1="94" y1="136" x2="64" y2="172" stroke="#f5f5f5" stroke-width="4"/>'
            '<line x1="94" y1="136" x2="124" y2="172" stroke="#f5f5f5" stroke-width="4"/>'
            '<line x1="94" y1="136" x2="94" y2="172" stroke="#f5f5f5" stroke-width="4"/>'
            '</g></svg>')


# ----------------------------------------------------------------------------- portada
if 'entrado' not in st.session_state:
    st.session_state.entrado = False


def entrar():
    st.session_state.entrado = True


def volver():
    st.session_state.entrado = False


if not st.session_state.entrado:
    try:
        with open(f'{DIR}/config.json') as fh:
            _cfg = json.load(fh)
        TEXTO_CATALOGO = f"{_cfg['n_peliculas']:,} películas".replace(',', ' ')
        if _cfg.get('n_recientes'):
            TEXTO_CATALOGO += f" · incluye estrenos hasta {_cfg['anio_max']}"
    except Exception:
        TEXTO_CATALOGO = '23 000+ películas'
    cinta()
    st.markdown(
        '<div class="portada">'
        f'<div class="camaras">{camara_svg(170)}{camara_svg(120, espejo=True)}</div>'
        '<div class="marquesina">Buscador de <span>películas</span></div>'
        '<div class="lema">Describe una escena, un personaje o una idea. Te mostramos la película.</div>'
        f'<div class="detalle">Búsqueda multimodal por póster y sinopsis · en español o en inglés · {TEXTO_CATALOGO}</div>'
        '</div>', unsafe_allow_html=True)
    _, centro, _ = st.columns([1, 1.5, 1])
    centro.button('Bienvenido · Entrar', on_click=entrar, width='stretch', key='entrar')
    cinta()
    st.markdown('<div class="creditos"><b>Michelle Ruiz · Diego Avella</b><br>Ingeniería Estadística<br>'
                'Estrenos: datos de TMDB (no respaldado ni certificado por TMDB)</div>',
                unsafe_allow_html=True)
    st.stop()


# ----------------------------------------------------------------------------- datos y modelos
@st.cache_resource(show_spinner='Cargando películas...')
def cargar_datos():
    cat = pd.read_parquet(f'{DIR}/catalogo.parquet')
    with open(f'{DIR}/config.json') as fh:
        cfg = json.load(fh)
    cargar = lambda n: np.load(f'{DIR}/{n}').astype(np.float32)
    V = {
        ('poster', 'Entrenada'): cargar('poster_adapt.npy'),
        ('sinopsis', 'Entrenada'): cargar('sinopsis_adapt.npy'),
        ('poster', 'Sin entrenar'): cargar('poster_orig.npy'),
        ('sinopsis', 'Sin entrenar'): cargar('sinopsis_orig.npy'),
    }
    assert all(len(v) == len(cat) for v in V.values())
    return cat, cfg, V


@st.cache_resource(show_spinner='Cargando los modelos de lenguaje (la primera vez tarda un poco)...')
def cargar_modelos(nombre_poster, nombre_sinopsis):
    from sentence_transformers import SentenceTransformer
    return SentenceTransformer(nombre_poster), SentenceTransformer(nombre_sinopsis)


cat, CFG, V = cargar_datos()
modelo_clip, modelo_e5 = cargar_modelos(CFG['modelo_poster'], CFG['modelo_sinopsis'])
ANIOS = cat['anio'].fillna(0).astype(int).values
ESTRENO = (cat['fuente'] == 'TMDB').values if 'fuente' in cat else np.zeros(len(cat), bool)


def z(s):
    return (s - s.mean()) / (s.std() + 1e-8)


CORRECCION = CFG.get('correccion', 'global')


def normalizar(s):
    # Estandariza los puntajes de una búsqueda. Si el catálogo mezcla MM-IMDB y TMDB, la corrección elegida en el
    # notebook 12 evita que una fuente salga de más solo porque sus vectores tienen otro nivel de similitud.
    if CORRECCION == 'global' or ESTRENO.all() or not ESTRENO.any():
        return z(s)
    out = np.empty_like(s)
    for m in (ESTRENO, ~ESTRENO):
        out[m] = z(s[m]) if CORRECCION == 'por_fuente' else s[m] - s[m].mean()
    return out if CORRECCION == 'por_fuente' else out / (out.std() + 1e-8)


@st.cache_data(max_entries=200, show_spinner=False)
def codificar(consulta):
    qp = modelo_clip.encode([consulta], convert_to_numpy=True)[0]
    qs = modelo_e5.encode([f'query: {consulta}'], convert_to_numpy=True, normalize_embeddings=True)[0]
    return qp / np.linalg.norm(qp), qs


def buscar(consulta, modo, version, alfa, desde, hasta, k, con_estrenos=True):
    qp, qs = codificar(consulta)
    zp = normalizar(V[('poster', version)] @ qp)
    zs = normalizar(V[('sinopsis', version)] @ qs)
    puntaje = {'Póster': zp, 'Sinopsis': zs}.get(modo, alfa * zp + (1 - alfa) * zs)
    puntaje = np.where((ANIOS >= desde) & (ANIOS <= hasta) & (con_estrenos | ~ESTRENO), puntaje, -np.inf)
    top = [int(i) for i in np.argsort(-puntaje)[:k] if np.isfinite(puntaje[i])]
    por_poster = {i: bool(alfa * zp[i] > (1 - alfa) * zs[i]) for i in top}
    return top, por_poster


def presentar_sinopsis(texto):
    # MM-IMDB trae las sinopsis en minúsculas: mayúscula al inicio de cada oración y en "i"
    texto = (texto or '').strip()
    if not texto:
        return 'Esta película no tiene sinopsis en el dataset.'
    texto = re.sub(r'\bi\b', 'I', texto)
    return re.sub(r'(^|[.!?]\s+)([a-záéíóúñ])', lambda m: m.group(1) + m.group(2).upper(), texto)


@st.dialog('Sinopsis', width='large')
def ver_sinopsis(i):
    r = cat.iloc[i]
    anio = int(r['anio']) if pd.notna(r['anio']) else '?'
    izq, der = st.columns([1, 3])
    with izq:
        st.image(Image.open(io.BytesIO(r['jpg'])), width=190)
    with der:
        st.markdown(f'### {html.escape(str(r["titulo"]))}')
        st.markdown(f'<div class="tarjeta-meta">{anio} · {html.escape((r["generos"] or "").replace(",", " · "))}</div>',
                    unsafe_allow_html=True)
        st.write('')
        if ESTRENO[i]:
            st.write(r.get('sinopsis', '') or 'Esta película no tiene sinopsis en TMDB.')
        else:
            st.write(presentar_sinopsis(r.get('sinopsis', '')))
    if ESTRENO[i]:
        st.caption('Póster y sinopsis de TMDB. Este producto usa la API de TMDB pero no está respaldado ni certificado por TMDB.')
    else:
        st.caption('Las sinopsis de MM-IMDB vienen en minúsculas y con los números reemplazados por "N".')


# ----------------------------------------------------------------------------- encabezado y pestañas
cab_izq, cab_der = st.columns([5, 1])
with cab_izq:
    st.markdown('<div class="marquesina" style="font-size:2.6rem; line-height:1; margin-top:0.3rem;">'
                '🎬 Buscador de <span style="color:#e50914">películas</span></div>', unsafe_allow_html=True)
with cab_der:
    st.button('← Inicio', on_click=volver, width='stretch')
cinta(delgada=True)

tab_buscar, tab_arq, tab_res, tab_tec = st.tabs(
    ['Buscador', 'Arquitectura', 'Resultados', 'Explicación técnica'])

# ============================================================================= BUSCADOR
with tab_buscar:
    EJEMPLOS = ['una casa embrujada en la nieve', 'a robot in space', 'princesas',
                'un detective investiga un asesinato', 'dibujos animados de colores', 'piratas en el mar']
    if 'consulta' not in st.session_state:
        st.session_state.consulta = ''

    def usar_ejemplo(texto):
        st.session_state.consulta = texto

    st.text_input('¿Qué película buscas? Escribe en español o en inglés', key='consulta',
                  placeholder='Ej.: una casa embrujada en la nieve')
    st.caption('O prueba con:')
    for fila in range(0, len(EJEMPLOS), 3):
        for c, ej in zip(st.columns(3), EJEMPLOS[fila:fila + 3]):
            c.button(ej, on_click=usar_ejemplo, args=(ej,), width='stretch', key=f'ej_{ej}')

    with st.expander('Opciones de búsqueda', expanded=False):
        o1, o2, o3 = st.columns([1.1, 1.2, 1.1])
        with o1:
            modo = st.radio('Buscar por', ['Ambos', 'Póster', 'Sinopsis'], horizontal=True)
            st.markdown('<div class="opcion-nota"><b>Póster:</b> lo que se ve. <b>Sinopsis:</b> lo que se cuenta. '
                        '<b>Ambos:</b> combina los dos; fue la forma que mejor buscó.</div>', unsafe_allow_html=True)
        with o2:
            alfa = st.slider('Peso del póster (modo "Ambos")', 0.0, 1.0, float(CFG['alfa']), 0.1,
                             disabled=(modo != 'Ambos'))
            st.markdown(f'<div class="opcion-nota">0 = solo sinopsis · 1 = solo póster. Viene en '
                        f'<b>{CFG["alfa"]:.1f}</b> porque fue el peso que <b>mostró los mejores resultados</b> '
                        f'en la evaluación (mAP 0.517 en películas de prueba, notebook 11).</div>',
                        unsafe_allow_html=True)
        with o3:
            version = st.radio('Representación', ['Entrenada', 'Sin entrenar'], horizontal=True)
            st.markdown('<div class="opcion-nota"><b>Entrenada:</b> vectores ajustados con nuestro adaptador. '
                        '<b>Sin entrenar:</b> los modelos tal cual. Cámbiala para ver el antes y el después.</div>',
                        unsafe_allow_html=True)
        o4, o5, o6 = st.columns([2, 1, 1])
        with o4:
            desde, hasta = st.slider('Años de estreno', CFG['anio_min'], CFG['anio_max'],
                                     (CFG['anio_min'], CFG['anio_max']))
        with o5:
            k = st.select_slider('Número de resultados', [6, 12, 18, 24], value=12)
        with o6:
            con_estrenos = st.toggle('Incluir estrenos 2020-2026', value=True, disabled=not ESTRENO.any())
            st.markdown('<div class="opcion-nota">Películas nuevas de TMDB que el modelo <b>nunca vio</b> al '
                        'entrenarse.</div>', unsafe_allow_html=True)

    consulta = st.session_state.consulta.strip()
    if not consulta:
        st.markdown('<div class="ficha"><p>Escribe una descripción o elige un ejemplo para empezar.</p></div>',
                    unsafe_allow_html=True)
    else:
        with st.spinner('Buscando...'):
            top, por_poster = buscar(consulta, modo, version, alfa, desde, hasta, k, con_estrenos)
        if not top:
            st.warning(f'No hay películas entre {desde} y {hasta}.')
        else:
            resumen = f'**{len(top)} películas** para *"{html.escape(consulta)}"*'
            if modo == 'Ambos':
                n_p = sum(por_poster.values())
                resumen += (f' · encontradas más por el póster: **{n_p}** · '
                            f'más por la sinopsis: **{len(top) - n_p}**')
            st.markdown(resumen)
            st.caption('Haz clic en "Ver sinopsis" debajo de cualquier película para leer de qué trata.'
                       + (' Las marcadas como ESTRENO vienen de TMDB.' if ESTRENO[top].any() else ''))
            COLUMNAS = 6
            for fila in range(0, len(top), COLUMNAS):
                for c, i in zip(st.columns(COLUMNAS), top[fila:fila + COLUMNAS]):
                    r = cat.iloc[i]
                    anio = int(r['anio']) if pd.notna(r['anio']) else '?'
                    generos = (r['generos'] or '').replace(',', ' · ')
                    with c:
                        st.image(Image.open(io.BytesIO(r['jpg'])), width='stretch')
                        tarjeta = (f'<div class="tarjeta-titulo">{html.escape(str(r["titulo"]))}</div>'
                                   f'<div class="tarjeta-meta">{anio} · {html.escape(generos)}</div>')
                        if modo == 'Ambos':
                            tarjeta += ('<span class="etiqueta">por el póster</span>' if por_poster[i]
                                        else '<span class="etiqueta sinopsis">por la sinopsis</span>')
                        if ESTRENO[i]:
                            tarjeta += ' <span class="etiqueta estreno">estreno</span>'
                        st.markdown(tarjeta, unsafe_allow_html=True)
                        if st.button('Ver sinopsis', key=f'sin_{i}', width='stretch'):
                            ver_sinopsis(i)

# ============================================================================= ARQUITECTURA
with tab_arq:
    st.markdown('## Cómo funciona el buscador')
    st.write('La búsqueda se hace por **dos caminos en paralelo**, uno por cada modalidad. En cada camino, '
             'la frase del usuario y las películas se convierten en vectores dentro del **mismo espacio**, y se '
             'buscan los más cercanos. Al final, los dos puntajes se combinan.')

    def caja(nombre, detalle, roja=False):
        return (f'<div class="caja{" roja" if roja else ""}"><div class="nombre">{nombre}</div>'
                f'<div class="det">{detalle}</div></div>')

    FL = '<div class="flecha">➜</div>'
    st.markdown(
        '<div class="carril">Camino 1 · el póster (lo que se ve)</div><div class="flujo">'
        + caja('Tu búsqueda', 'texto en español o inglés')
        + FL + caja('CLIP multilingüe', 'DistilBERT multilingüe (6 capas) → promedio → capa lineal 768→512')
        + FL + caja('Similitud coseno', 'contra el vector de cada póster')
        + '</div><div class="flujo">'
        + caja('Póster', 'imagen 224×224 dividida en parches de 32×32')
        + FL + caja('CLIP ViT-B/32', 'Vision Transformer de 12 capas → vector de 512 (congelado)')
        + FL + caja('Adaptador', 'capa lineal residual entrenada con los géneros', roja=True)
        + '</div>'
        '<div class="carril">Camino 2 · la sinopsis (lo que se cuenta)</div><div class="flujo">'
        + caja('Tu búsqueda', 'con el prefijo "query:"')
        + FL + caja('multilingual-e5-base', 'XLM-RoBERTa (12 capas) → promedio → vector de 768')
        + FL + caja('Similitud coseno', 'contra el vector de cada sinopsis')
        + '</div><div class="flujo">'
        + caja('Sinopsis', 'texto completo, hasta 512 tokens, prefijo "passage:"')
        + FL + caja('multilingual-e5-base', 'el mismo modelo, congelado → vector de 768')
        + FL + caja('Adaptador', 'capa lineal residual entrenada con los géneros', roja=True)
        + '</div>'
        '<div class="carril">Combinación</div><div class="flujo">'
        + caja('Estandarizar', 'cada puntaje se pasa a z-score para que estén en la misma escala')
        + FL + caja('Mezclar', f'α · póster + (1 − α) · sinopsis, con α = {CFG["alfa"]:.1f}', roja=True)
        + FL + caja('Ranking', 'las películas con mayor puntaje, filtradas por año')
        + '</div>', unsafe_allow_html=True)

    st.markdown('### Los modelos, uno por uno')
    fichas = [
        ('CLIP ViT-B/32 (OpenAI)',
         'Entrenado con 400 millones de pares imagen-texto para que una foto y su descripción queden cerca. '
         'Su parte de imagen es un <b>Vision Transformer</b>: corta el póster en 49 parches de 32×32 píxeles, '
         'los trata como "palabras" y los procesa con 12 capas de atención. Resultado: 512 números por póster.'),
        ('CLIP multilingüe (sentence-transformers)',
         'El CLIP original solo entiende inglés. Esta versión es un <b>DistilBERT multilingüe</b> entrenado por '
         '<b>destilación</b>: aprendió a imitar al codificador de texto de CLIP en más de 50 idiomas. Por eso '
         '"una película de terror" y "a horror movie" caen casi en el mismo punto (similitud 0.99).'),
        ('multilingual-e5-base (Microsoft)',
         'Un <b>XLM-RoBERTa</b> de 12 capas entrenado para <b>buscar documentos largos a partir de frases cortas</b> '
         'en ~100 idiomas. Lo usamos para la sinopsis porque el texto de CLIP solo sirve para frases cortas: con CLIP, '
         'todas las sinopsis "se parecían igual" a cualquier búsqueda.'),
        ('Adaptador residual (nuestro)',
         'Una capa lineal que <b>corrige un poco</b> el vector de cada película: v′ = normalizar(v + W·v). '
         'W empieza en cero (al inicio no cambia nada) y se entrena para que las películas queden cerca de las '
         'frases de sus géneros. Hay uno para el póster (512×512) y otro para la sinopsis (768×768). '
         'Las búsquedas no se modifican, por eso cualquier frase libre sigue funcionando.'),
        ('Clasificadores de género (notebooks 04 y 06)',
         'En la otra parte del proyecto, sobre los mismos vectores congelados (CLIP para el póster, DistilBERT para '
         'la sinopsis) se entrenan cabezas pequeñas que predicen los 23 géneros: solo póster, solo sinopsis y la '
         'fusión de ambos. Sirven para medir de qué modalidad depende el modelo.'),
    ]
    a1, a2 = st.columns(2)
    for j, (titulo, texto) in enumerate(fichas):
        (a1 if j % 2 == 0 else a2).markdown(f'<div class="ficha"><h4>{titulo}</h4><p>{texto}</p></div>',
                                            unsafe_allow_html=True)
    st.caption('Todos los modelos preentrenados se usan congelados: solo se entrenan los adaptadores y las cabezas, '
               'por eso todo el proyecto corre en Google Colab.')

# ============================================================================= RESULTADOS
with tab_res:
    def barras(df, x, y, color, orden_color, titulo, horizontal=False, alto=280):
        esc = alt.Scale(domain=orden_color, range=[GRIS, ROJO, BLANCO][:len(orden_color)])
        if horizontal:
            enc = dict(y=alt.Y(f'{y}:N', sort='-x', title=None), x=alt.X(f'{x}:Q', title=None))
        else:
            enc = dict(x=alt.X(f'{x}:N', title=None, sort=None, axis=alt.Axis(labelAngle=0)),
                       y=alt.Y(f'{y}:Q', title=None), xOffset=alt.XOffset(f'{color}:N', sort=orden_color))
        ch = (alt.Chart(df, title=titulo).mark_bar()
              .encode(color=alt.Color(f'{color}:N', scale=esc, sort=orden_color,
                                      legend=alt.Legend(orient='top', title=None)),
                      tooltip=list(df.columns), **enc)
              .properties(height=alto))
        return (ch.configure(background='#000')
                  .configure_view(stroke=None)
                  .configure_axis(labelColor='#cfcfcf', titleColor='#cfcfcf', gridColor='#1f1f1f', domainColor='#444')
                  .configure_legend(labelColor='#cfcfcf')
                  .configure_title(color='#f5f5f5', font='Inter', fontSize=14, anchor='start'))

    st.markdown('## Lo que encontramos')
    k1, k2, k3, k4 = st.columns(4)
    for col, valor, texto in [(k1, '0.27 → 0.51', 'mAP del buscador antes y después de entrenar'),
                              (k2, '7.7 / 10', 'aciertos en los 10 primeros resultados (antes: 5.6)'),
                              (k3, '22 / 22', 'géneros en los que mejora la búsqueda'),
                              (k4, '63.6', 'F1 macro al clasificar géneros con póster + sinopsis')]:
        col.markdown(f'<div class="kpi"><div class="valor">{valor}</div><div class="texto">{texto}</div></div>',
                     unsafe_allow_html=True)
    st.write('')

    st.markdown('### 1. El entrenamiento mejora la búsqueda')
    antes_despues = pd.DataFrame({
        'modo': ['Póster', 'Póster', 'Sinopsis', 'Sinopsis', 'Ambos', 'Ambos'],
        'etapa': ['Sin entrenar', 'Entrenada'] * 3,
        'mAP': [0.209, 0.374, 0.226, 0.452, 0.272, 0.513]})
    r1, r2 = st.columns([1.3, 1])
    r1.altair_chart(barras(antes_despues, 'modo', 'mAP', 'etapa', ['Sin entrenar', 'Entrenada'],
                           'mAP en películas de prueba (7 090 que el modelo nunca vio)'), width='stretch', theme=None)
    r2.markdown('<div class="ficha"><h4>Qué significa</h4><p>El <b>mAP</b> mide qué tan arriba quedan las películas '
                'correctas (1 = perfecto, azar ≈ 0.11). Después de entrenar el adaptador, buscar con póster y sinopsis '
                'juntos pasa de <b>0.27 a 0.51</b>. La mejora es real: su intervalo de confianza del 95 % va de '
                '+0.20 a +0.28, lejos de cero, y es igual con 3 semillas distintas.</p></div>',
                unsafe_allow_html=True)

    st.markdown('### 2. Combinar póster y sinopsis es mejor que cualquiera por separado')
    modos = pd.DataFrame({
        'forma de buscar': ['Póster (CLIP)', 'Sinopsis (CLIP)', 'Sinopsis (E5)', 'Ambos (CLIP)', 'Ambos (E5)'],
        'mAP': [0.204, 0.121, 0.222, 0.188, 0.263], 'tipo': ['sin entrenar'] * 5})
    r1, r2 = st.columns([1.3, 1])
    r1.altair_chart(barras(modos, 'mAP', 'forma de buscar', 'tipo', ['sin entrenar'],
                           'mAP sin entrenar, según cómo se representa cada modalidad', horizontal=True, alto=230)
                    .encode(color=alt.value(ROJO)), width='stretch', theme=None)
    r2.markdown('<div class="ficha"><h4>El desbalance depende de la representación</h4><p>Cuando la sinopsis estaba mal '
                'representada (texto de CLIP), el mejor peso para el póster era <b>0.9</b>: la combinación casi '
                'ignoraba la sinopsis. Con una buena representación (E5) el mejor peso bajó a <b>0.5</b>, y después '
                'de entrenar a <b>0.4</b>: las dos modalidades aportan. <b>El desbalance no es culpa de la '
                'modalidad, sino de cómo la representamos.</b></p></div>', unsafe_allow_html=True)

    st.markdown('### 3. Dónde mejora más')
    genero = pd.DataFrame({
        'género': ['Animation', 'Documentary', 'Action', 'Western', 'Adventure', 'Horror', 'Family', 'Music',
                   'Drama', 'Sport', 'Comedy', 'Crime', 'Mystery', 'Biography', 'Romance', 'History', 'Sci-Fi',
                   'War', 'Thriller', 'Musical', 'Fantasy', 'Film-Noir'],
        'mejora en AP': [0.460, 0.435, 0.417, 0.320, 0.319, 0.315, 0.259, 0.256, 0.228, 0.224, 0.217, 0.216,
                         0.209, 0.207, 0.185, 0.185, 0.179, 0.160, 0.158, 0.120, 0.117, 0.112],
        'tipo': ['mejora'] * 22})
    r1, r2 = st.columns([1.3, 1])
    r1.altair_chart(barras(genero, 'mejora en AP', 'género', 'tipo', ['mejora'],
                           'Mejora por género al entrenar (póster + sinopsis)', horizontal=True, alto=470)
                    .encode(color=alt.value(ROJO)), width='stretch', theme=None)
    with r2:
        st.markdown('<div class="ficha"><h4>Cada modalidad para lo suyo</h4><p>Sin entrenar, el <b>póster</b> gana en '
                    'los géneros que se <i>ven</i>: animación, western, acción, terror, familia. La <b>sinopsis</b> gana '
                    'en los que se <i>cuentan</i>: documental (0.49 contra 0.12 del póster), música, biografía, guerra '
                    'e historia.</p></div>', unsafe_allow_html=True)
        st.markdown('<div class="ficha"><h4>Idioma</h4><p>El buscador funciona en los dos idiomas: mAP 0.283 en '
                    'inglés y 0.244 en español. Escribir una sola palabra ("terror") funciona casi igual que una '
                    'frase completa.</p></div>', unsafe_allow_html=True)
        st.markdown('<div class="ficha"><h4>Clasificación de géneros</h4><p>F1 macro en prueba, 5 semillas: '
                    'solo póster <b>47.7</b>, solo sinopsis <b>59.1</b>, fusión <b>63.6</b>. La imagen sí aporta '
                    '(+4.5 puntos sobre la sinopsis sola).</p></div>', unsafe_allow_html=True)

    st.markdown('### 4. El espacio de representaciones antes y después')
    if os.path.exists(f'{DIR}/umap.png'):
        st.image(f'{DIR}/umap.png', width='stretch')
    st.write('Cada punto es una película de prueba, coloreada por su género. **Antes** de entrenar los géneros '
             'están mezclados; **después** se agrupan. Con números: de los 10 vecinos más cercanos de cada película, '
             'la proporción que comparte algún género sube de 0.71 a 0.76 en el póster y de 0.76 a 0.83 en la '
             'sinopsis (el azar da 0.54).')

    st.markdown('### 5. Películas de ayer y de hoy')
    TEMP = CFG.get('temporal', {})
    if not TEMP:
        st.markdown('<div class="ficha"><h4>En construcción</h4><p>La comparación entre películas de MM-IMDB (hasta 2016) '
                    'y películas de 2020 a 2026 (notebooks 06 a 08 y 12) se agregará aquí.</p></div>', unsafe_allow_html=True)
    else:
        st.write('Todo se entrenó con películas de MM-IMDB (hasta 2016) y se probó, **sin reentrenar**, con 3 000 '
                 'películas de 2020 a 2026 descargadas de TMDB. Para separar el efecto de la **época** del de la '
                 '**fuente** de los datos, se compara contra un grupo de control: películas viejas cuyo póster y '
                 'sinopsis también se descargaron de TMDB.')
        kpis = []
        if 'clasificacion' in TEMP:
            c_ = TEMP['clasificacion']
            kpis += [(f"{100 * c_['ap_texto']['diferencia']:+.1f}", 'puntos de AP pierde la sinopsis al clasificar géneros de películas nuevas'),
                     (f"{100 * c_['ap_imagen']['diferencia']:+.1f}", 'puntos de AP cambia el póster (prácticamente nada)')]
        if 'dependencia' in TEMP and {'control_tmdb', 'recientes_tmdb'} <= set(TEMP['dependencia']):
            d_ = TEMP['dependencia']
            kpis.append((f"{d_['control_tmdb']:.2f} → {d_['recientes_tmdb']:.2f}",
                         'cuánto depende la clasificación del póster (control → recientes)'))
        if 'busqueda' in TEMP:
            m_ = TEMP['busqueda']['map_despues']
            kpis.append((f"{m_['control_tmdb']['ambos']:.2f} → {m_['recientes_tmdb']['ambos']:.2f}",
                         'mAP del buscador con póster + sinopsis (control → recientes)'))
        for col, (valor, texto) in zip(st.columns(max(len(kpis), 1)), kpis):
            col.markdown(f'<div class="kpi"><div class="valor">{valor}</div><div class="texto">{texto}</div></div>',
                         unsafe_allow_html=True)
        st.write('')
        filas_t = []
        if 'clasificacion' in TEMP:
            for m_, nombre in [('ap_imagen', 'Póster'), ('ap_texto', 'Sinopsis'), ('ap_fusion', 'Ambos')]:
                filas_t.append({'tarea': 'Clasificar géneros', 'modalidad': nombre,
                                'cambio (puntos)': round(100 * TEMP['clasificacion'][m_]['diferencia'], 1)})
        if 'busqueda' in TEMP:
            for m_, nombre in [('poster', 'Póster'), ('sinopsis', 'Sinopsis'), ('ambos', 'Ambos')]:
                filas_t.append({'tarea': 'Buscar por género', 'modalidad': nombre,
                                'cambio (puntos)': round(100 * TEMP['busqueda']['epoca'][m_]['diferencia'], 1)})
        r1, r2 = st.columns([1.3, 1])
        if filas_t:
            r1.altair_chart(barras(pd.DataFrame(filas_t), 'tarea', 'cambio (puntos)', 'modalidad',
                                   ['Póster', 'Sinopsis', 'Ambos'], 'Cambio con la época: recientes − control (AP)'),
                            width='stretch', theme=None)
        r2.markdown('<div class="ficha"><h4>Qué significa</h4><p>Con películas nuevas la <b>sinopsis</b> pierde '
                    'capacidad de predecir el género y el <b>póster</b> se mantiene. Como el póster pesa lo mismo, '
                    'el modelo pasa a <b>depender más de la imagen</b>: el desbalance entre modalidades no es fijo, '
                    'cambia con la época de las películas.</p></div>', unsafe_allow_html=True)
        st.caption('Estrenos: datos e imágenes de TMDB. Este producto usa la API de TMDB pero no está respaldado ni '
                   'certificado por TMDB.')

# ============================================================================= EXPLICACIÓN TÉCNICA
with tab_tec:
    st.markdown('## Qué hicimos, explicado')
    st.write('Los conceptos del curso detrás del proyecto, de lo más general a lo más técnico.')

    with st.expander('1 · Representaciones: convertir películas en números', expanded=True):
        st.markdown(
            'Una **representación** (o *embedding*) es un vector de números que resume un dato. Un buen modelo pone '
            'cosas parecidas en vectores cercanos: dos pósters de terror quedan cerca entre sí y lejos de uno de '
            'dibujos animados. Todo el proyecto ocurre en esos espacios: **buscar es encontrar los vectores más '
            'cercanos** a la búsqueda.\n\n'
            'Usamos modelos **preentrenados y congelados**: ya aprendieron a representar imágenes y textos con '
            'millones de ejemplos, y nosotros solo entrenamos capas pequeñas encima (*transfer learning*). '
            'Por eso todo corre en Google Colab.')

    with st.expander('2 · Las redes neuronales: Transformers y atención'):
        st.markdown(
            'Todos los modelos que usamos son **Transformers**. Su pieza clave es la **atención**: cada palabra '
            '(o cada parche de imagen) mira a todas las demás y decide cuáles le importan para entender su significado.')
        st.latex(r'\text{Atención}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right) V')
        st.markdown(
            '- **Q, K, V** (consultas, claves, valores) son proyecciones de cada elemento. $QK^\\top$ mide qué tanto '
            'se relaciona cada par, y el softmax lo convierte en pesos que suman 1.\n'
            '- **Vision Transformer (ViT):** corta la imagen en parches de 32×32 y los trata como palabras.\n'
            '- **BERT / DistilBERT / XLM-RoBERTa:** Transformers de texto. DistilBERT es una versión reducida '
            '(6 capas en vez de 12) que conserva casi todo el rendimiento.\n'
            '- **Destilación:** un modelo "alumno" aprende a imitar las salidas de un "maestro". Así se hizo el '
            'CLIP multilingüe: aprendió a producir los mismos vectores que CLIP, pero en más de 50 idiomas.')

    with st.expander('3 · Aprendizaje contrastivo: cómo CLIP une imágenes y textos'):
        st.markdown(
            'CLIP se entrenó con pares (imagen, texto). Para cada lote, la pérdida **InfoNCE** premia que cada imagen '
            'quede cerca de *su* texto y lejos de los textos de las demás:')
        st.latex(r'\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}\log\frac{\exp(\text{sim}(x_i, t_i)/\tau)}'
                 r'{\sum_{j=1}^{N}\exp(\text{sim}(x_i, t_j)/\tau)}')
        st.markdown(
            'Por eso un texto y una imagen se pueden comparar directamente: viven en el mismo espacio. '
            'Un efecto curioso que encontramos: CLIP también aprendió a **leer texto en las imágenes**, así que a '
            'veces encuentra un póster porque la palabra buscada está escrita en él (*Murder!* para "asesinato").')

    with st.expander('4 · Búsqueda semántica, brecha modal y z-score'):
        st.markdown('La cercanía entre la búsqueda $q$ y una película $v$ se mide con la **similitud coseno**:')
        st.latex(r'\text{sim}(q, v) = \frac{q \cdot v}{\lVert q \rVert\,\lVert v \rVert}')
        st.markdown(
            'Problema: texto contra texto da similitudes más altas que texto contra imagen. Es la **brecha modal**: '
            'en CLIP, imágenes y textos ocupan zonas distintas del espacio. Si se sumaran tal cual, la sinopsis '
            'dominaría siempre. Por eso cada puntaje se **estandariza** sobre todo el catálogo antes de combinarlos:')
        st.latex(r'\text{puntaje} = \alpha\, z_{\text{póster}} + (1-\alpha)\, z_{\text{sinopsis}}, '
                 r'\qquad z = \frac{s - \bar{s}}{\sigma_s}')

    with st.expander('5 · El adaptador: cómo entrenamos la representación'):
        st.markdown('Una capa lineal **residual** que corrige el vector de cada película:')
        st.latex(r"v' = \frac{v + Wv}{\lVert v + Wv \rVert}, \qquad W_{\text{inicial}} = 0")
        st.markdown(
            'Para cada película y cada género $g$ se calcula un puntaje $\\ell_g = s \\cdot \\text{sim}(v\', q_g) + b_g$, '
            'donde $q_g$ es una frase del género ("cine de terror", "a horror story"). Se entrena con **entropía '
            'cruzada binaria multietiqueta** (una película puede tener varios géneros) más un castigo que evita que '
            '$W$ se aleje demasiado:')
        st.latex(r'\mathcal{L} = -\frac{1}{G}\sum_{g}\big[y_g \log \sigma(\ell_g) + (1-y_g)\log(1-\sigma(\ell_g))\big]'
                 r' + \lambda \lVert W \rVert^2')
        st.markdown(
            '- **Optimizador Adam**, lotes de 256, **recorte de gradiente** para evitar saltos.\n'
            '- **Early stopping:** se detiene cuando la pérdida en validación deja de bajar.\n'
            '- **3 semillas** para comprobar que el resultado no depende del azar.\n'
            '- **Sin trampa:** se entrena con películas de *train*, se ajusta con *dev* y se evalúa solo con *test* '
            '(películas que nunca vio), y con frases distintas a las de evaluación.')

    with st.expander('6 · Métricas: cómo medimos'):
        st.markdown(
            '- **precision@10:** de los 10 primeros resultados, ¿qué fracción es del género buscado? Es lo que ve el usuario.\n'
            '- **AP (average precision):** promedia la precisión en cada posición donde aparece una película correcta. '
            'Premia ponerlas arriba. **mAP** es su promedio sobre todas las búsquedas. 1 = perfecto; el azar da '
            'la proporción de películas del género.')
        st.latex(r'\text{AP} = \frac{1}{R}\sum_{k=1}^{n} P(k)\cdot \text{rel}(k)')
        st.markdown(
            '- **F1 macro** (clasificación): media armónica de precisión y cobertura, calculada para cada género y '
            'promediada sin ponderar, así los géneros raros pesan igual que los comunes.\n'
            '- **Intervalo de confianza bootstrap:** se remuestrean los géneros con reemplazo 2 000 veces y se mira '
            'dónde cae el 95 % de los resultados. Si el intervalo de la mejora no incluye el cero, la mejora es real.\n'
            '- **Pureza de vecinos:** de los 10 vecinos más cercanos de cada película, ¿cuántos comparten algún '
            'género? Mide si el espacio quedó mejor organizado.\n'
            '- **UMAP:** proyecta los vectores de 512 o 768 dimensiones a 2D para poder verlos, intentando conservar '
            'quién está cerca de quién.')

    with st.expander('7 · La pregunta del proyecto: el desbalance entre modalidades'):
        st.markdown(
            'En un modelo que combina imagen y texto, una de las dos puede **dominar** y la otra quedar casi sin usar '
            '(*greedy modality*). Lo medimos de dos formas:\n\n'
            '1. **Clasificación:** comparando solo póster, solo sinopsis y la fusión (la fusión gana: la imagen aporta).\n'
            '2. **Búsqueda:** buscando el mejor peso α entre póster y sinopsis. Con la sinopsis mal representada, el '
            'mejor α era 0.9 (la sinopsis casi se ignoraba). Con buenas representaciones, 0.4-0.5.\n\n'
            '**Conclusión:** el desbalance depende de la calidad con la que se representa cada modalidad. '
            'La otra mitad del proyecto pregunta si ese equilibrio se mantiene con **películas recientes** (2020-2026): '
            'con películas nuevas la sinopsis pierde más que el póster (pestaña Resultados, sección 5).')
'''

with open(f'{APP_DIR}/app.py', 'w') as fh:
    fh.write(APP)

os.makedirs(f'{APP_DIR}/.streamlit', exist_ok=True)
with open(f'{APP_DIR}/.streamlit/config.toml', 'w') as fh:
    fh.write('[theme]\nbase = "dark"\nprimaryColor = "#e50914"\n'
             'backgroundColor = "#000000"\nsecondaryBackgroundColor = "#121212"\ntextColor = "#f5f5f5"\n\n'
             '[server]\nheadless = true\n\n[browser]\ngatherUsageStats = false\n\n'
             '[client]\ntoolbarMode = "minimal"\n')

with open(f'{APP_DIR}/requirements.txt', 'w') as fh:
    fh.write('--extra-index-url https://download.pytorch.org/whl/cpu\n'
             'torch\nsentence-transformers\nstreamlit\npandas\npyarrow\nnumpy\npillow\n')

with open(f'{APP_DIR}/Dockerfile', 'w') as fh:
    fh.write('FROM python:3.11-slim\n'
             'WORKDIR /app\n'
             'COPY requirements.txt .\n'
             'RUN pip install --no-cache-dir -r requirements.txt\n'
             'COPY . .\n'
             'ENV HOME=/tmp HF_HOME=/tmp/hf\n'
             'EXPOSE 8501\n'
             'CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]\n')

print('Archivos de la app:', sorted(os.listdir(APP_DIR)))

Archivos de la app: ['.streamlit', 'Dockerfile', 'app.py', 'catalogo.parquet', 'config.json', 'poster_adapt.npy', 'poster_orig.npy', 'requirements.txt', 'sinopsis_adapt.npy', 'sinopsis_orig.npy', 'umap.png']


## 3. Chequeo: ¿la app busca igual que el notebook 11?

Se repite la búsqueda *"una película de terror"* con la misma lógica de la app y se cuenta cuántos de los 12 primeros
resultados son de terror, con la representación sin entrenar y con la entrenada. Con la entrenada debería haber más.

In [5]:
from sentence_transformers import SentenceTransformer

modelo_clip = SentenceTransformer(config['modelo_poster'])
modelo_e5 = SentenceTransformer(config['modelo_sinopsis'])
V = {n[:-4]: np.load(f'{APP_DIR}/{n}').astype(np.float32) for n in vectores}

def z(s):
    return (s - s.mean()) / (s.std() + 1e-8)

def buscar_prueba(consulta, version, alfa=config['alfa'], k=12):
    qp = modelo_clip.encode([consulta])[0]; qp = qp / np.linalg.norm(qp)
    qs = modelo_e5.encode([f'query: {consulta}'], normalize_embeddings=True)[0]
    s = alfa * z(V[f'poster_{version}'] @ qp) + (1 - alfa) * z(V[f'sinopsis_{version}'] @ qs)
    return cat.iloc[np.argsort(-s)[:k]]

for version in ['orig', 'adapt']:
    r = buscar_prueba('una película de terror', version)
    n = r['generos'].str.contains('Horror').sum()
    print(f"{'sin entrenar' if version == 'orig' else 'entrenada':>12}: {n}/12 de terror | "
          f"{', '.join(r['titulo'].head(4))}")

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.65k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/572 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  539MB            

... (21 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

sin entrenar: 12/12 de terror | The Fear, Midnight Movie, Darkest Night, The Exorcism
   entrenada: 12/12 de terror | The Fear, The Shrine, Kalinga, Bloodbath at the House of Death


## 4. Lanzar la app desde Colab

Streamlit corre dentro de Colab y **Cloudflare** crea un link público temporal (`https://….trycloudflare.com`)
para abrirla desde cualquier navegador, incluso desde el celular. El link funciona mientras esta sesión de Colab
esté abierta.

La primera búsqueda tarda unos segundos porque la app carga los modelos; después es casi instantánea.

In [6]:
# Descargar cloudflared (solo la primera vez en la sesión)
if not os.path.exists('/content/cloudflared'):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

# Detener una versión anterior de la app, si la había
for patron in ['streamlit run', 'cloudflared tunnel']:
    subprocess.run(['pkill', '-f', patron])
time.sleep(1)

env = dict(os.environ, BUSCADOR_DIR=APP_DIR)
app_log = open('/content/streamlit.log', 'w')
subprocess.Popen(['streamlit', 'run', f'{APP_DIR}/app.py', '--server.port', '8501', '--server.headless', 'true'],
                 cwd=APP_DIR, env=env, stdout=app_log, stderr=subprocess.STDOUT)

tunel_log = open('/content/cloudflared.log', 'w')
subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8501', '--no-autoupdate'],
                 stdout=tunel_log, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/cloudflared.log').read())
    if m:
        url = m.group(0); break

if url:
    print('La app está en:', url)
    print('(si al abrirla aparece un error, espera unos 20 segundos y recarga: la app todavía está arrancando)')
else:
    print('No apareció el link. Revisa los registros:')
    print(open('/content/cloudflared.log').read()[-1500:])
    print(open('/content/streamlit.log').read()[-1500:])

No apareció el link. Revisa los registros:
2026-11-12T04:39:32Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-12T04:39:32Z INF Requesting new quick Tunnel on trycloudflare.com...
quick tunnel provisioning failed with status 429: [10000] Tunnelstore request failed

2026-11-12 04:39:36.484 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  E

In [7]:
# Si algo falla en la app, aquí se ven los errores
print(open('/content/streamlit.log').read()[-3000:])

2026-11-12 04:39:36.484 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.194.145.21:8501




## 5. (Opcional) Publicarla de forma permanente en Hugging Face Spaces

El link de Cloudflare muere cuando se cierra Colab. Para tener un **link fijo** (por ejemplo para la presentación
o para integrarlo en la página final del proyecto) se puede publicar gratis en **Hugging Face Spaces**:

1. Crea una cuenta en [huggingface.co](https://huggingface.co) (gratis).
2. Crea un token con permiso de escritura: *Settings → Access Tokens → Create new token → Write*.
3. En Colab, abre el panel de la **llave** 🔑 (Secretos, a la izquierda), agrega uno llamado `HF_TOKEN` con ese token
   y activa "Acceso al notebook".
4. Cambia `USUARIO_HF` abajo por tu usuario y corre la celda.

La subida tarda unos minutos (~350 MB). Después, Hugging Face construye la app (~5-10 minutos la primera vez);
el avance se ve en la pestaña *Logs* del Space.

In [8]:
USUARIO_HF = 'tu-usuario'          # <- cambia esto
NOMBRE_SPACE = 'buscador-peliculas'
PUBLICAR = False                   # <- cambia a True para publicar

if PUBLICAR:
    from huggingface_hub import HfApi
    from google.colab import userdata
    api = HfApi(token=userdata.get('HF_TOKEN'))
    repo_id = f'{USUARIO_HF}/{NOMBRE_SPACE}'
    api.create_repo(repo_id, repo_type='space', space_sdk='docker', exist_ok=True)

    readme = ('---\ntitle: Buscador de películas\nemoji: 🎬\ncolorFrom: yellow\ncolorTo: gray\n'
              'sdk: docker\napp_port: 8501\npinned: false\n---\n\n'
              'Buscador semántico de películas (póster + sinopsis) del proyecto '
              '[Proyecto_Multimodal](https://github.com/AndresAvella09/Proyecto_Multimodal).\n')
    with open(f'{APP_DIR}/README.md', 'w') as fh:
        fh.write(readme)

    api.upload_folder(folder_path=APP_DIR, repo_id=repo_id, repo_type='space')
    print('Publicado en: https://huggingface.co/spaces/' + repo_id)
else:
    print('No se publicó (PUBLICAR = False).')

No se publicó (PUBLICAR = False).


## Resumen

- La carpeta `app_buscador/` en Drive tiene **todo** lo que necesita la app: datos, vectores, código y configuración.
- La app se puede abrir desde Colab (sección 4) o publicar con un link fijo (sección 5).
- Más adelante, la misma app puede crecer con más páginas de Streamlit (mapa de películas, resultados de la comparación temporal)
  para armar la página final del proyecto.